# 22-28 · Перенос данных через JSON

Практика к разделу [«Перенос данных между базами и роль JSON»](../../site/chapters/glava-22/22-28-perenos-dannyh.html).

## Цель

Перенести три строки из одной базы SQLite в другую через JSON как промежуточный формат — и проверить, что количество и содержимое совпали.

## Рабочий пример — исходная база

In [1]:
import json
import sqlite3

istochnik = sqlite3.connect(":memory:")
istochnik.row_factory = sqlite3.Row
istochnik.execute("CREATE TABLE tasks (id INTEGER PRIMARY KEY, title TEXT NOT NULL, done INTEGER NOT NULL DEFAULT 0)")
istochnik.executemany(
    "INSERT INTO tasks (title, done) VALUES (?, ?)",
    [("Купить хлеб", 0), ("Выучить SQL", 1), ("Собрать сайт", 0)],
)
istochnik.commit()

stroki = istochnik.execute("SELECT title, done FROM tasks ORDER BY id").fetchall()
dannye = [dict(stroka) for stroka in stroki]
tekst_json = json.dumps(dannye, ensure_ascii=False)
print(tekst_json)

[{"title": "Купить хлеб", "done": 0}, {"title": "Выучить SQL", "done": 1}, {"title": "Собрать сайт", "done": 0}]


## Эксперимент — импорт в новую базу

In [2]:
zagruzhennye = json.loads(tekst_json)

cel = sqlite3.connect(":memory:")
cel.execute("CREATE TABLE tasks (id INTEGER PRIMARY KEY, title TEXT NOT NULL, done INTEGER NOT NULL DEFAULT 0)")
for zapis in zagruzhennye:
    cel.execute("INSERT INTO tasks (title, done) VALUES (?, ?)", (zapis["title"], zapis["done"]))
cel.commit()

itog = cel.execute("SELECT title, done FROM tasks ORDER BY id").fetchall()
print(itog)

[('Купить хлеб', 0), ('Выучить SQL', 1), ('Собрать сайт', 0)]


## Проверка результата

In [3]:
iskhodnye_nazvaniya = [s["title"] for s in stroki]
itogovye_nazvaniya = [s[0] for s in itog]

assert len(itog) == len(stroki) == 3
assert itogovye_nazvaniya == iskhodnye_nazvaniya
assert [s[1] for s in itog] == [s["done"] for s in stroki]
print("Верно: количество строк и их содержимое совпадают после переноса через JSON.")

Верно: количество строк и их содержимое совпадают после переноса через JSON.


## Важная деталь

Новые идентификаторы `id` в целевой базе назначаются заново — при вставке они не передавались явно. Это осознанный выбор для такого простого случая: если бы исходные `id` были важны (например, на них кто-то уже ссылается), их нужно было бы переносить явно и после этого проверить, что они остались уникальными.